# 02 — Airfoil selection

NeuralFoil is a section-model screening tool. Compare at the selected design
Reynolds number and cruise lift coefficient; 3D trim, roughness and structural
thickness still determine the actual aircraft. S5010 provides a reflexed root;
E186 is the selected tip candidate and its required twist is checked with VLM in
notebook 04. A section L/D ranking alone does not select a flying-wing airfoil.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=False)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, dpi=140)
    plt.show()


In [ ]:
from uavdesign.aero import airfoil as af
re = d.selected.polar_info["reynolds"]
cl = d.selected.summary()["cl_cruise"]
summaries = [af.summarize(name, re, cl).as_dict() for name in studies["airfoil_candidates"]]
display(pd.DataFrame(summaries))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name in studies["airfoil_candidates"]:
    polar = af.section_polar(name, re)
    axes[0].plot(polar["cl"], polar["cd"], label=name)
    axes[1].plot(polar["cl"], polar["cm"], label=name)
axes[0].set(xlabel="Section lift coefficient", ylabel="Section drag coefficient")
axes[1].set(xlabel="Section lift coefficient", ylabel="Section pitching-moment coefficient")
for ax in axes: ax.grid(alpha=.3); ax.legend()
save(fig, "02_airfoil_comparison.png")
print(f"Selected root={pl['airfoil_root']}, tip={pl['airfoil_tip']}; Re={re:.0f}; cruise CL={cl:.3f}")
